# 06 Лаборатория — Временные спреды: календари, диагонали и PMCC

Эта лаборатория строит каждую конструкцию временного спреда на цепочке DEMO (спот 100) и показывает,
почему позиции со смешанными экспирациями **обязаны** оцениваться через `payoff.pnl_at` на дату
экспирации ближней ноги, а не наивной диаграммой внутренней стоимости на экспирации. Вы:

1. Соберёте ATM-календарь (calendar spread) и нарисуете его настоящую «палатку» P&L на экспирации
   ближней ноги.
2. Измерите знак веги календаря сеткой сценариев со сдвигом волатильности.
3. Соберёте бычью диагональ (diagonal spread) и «бедняцкий покрытый колл» (poor man's covered call,
   PMCC) и сделаете по ним сводку.

Запускайте сверху вниз. Ничто здесь не обращается к сети.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, greeks, pricing, data

chain = data.load_sample_chain("DEMO")
SPOT = 100.0
chain.head()

## 1. ATM-календарь

Продаём 100-й колл (call) на 21 DTE (середина рынка 2.66), покупаем 100-й колл на 45 DTE (3.91). Тот
же страйк, тот же тип, разные экспирации. Экспирации передаются **в годах**.

In [ ]:
cal = strategies.calendar_spread(
    "call", 100.0,
    front_expiry=21/365, front_premium=2.66,
    back_expiry=45/365, back_premium=3.91,
)
print(cal.describe())
print("чистая премия ($, + = дебет):", cal.net_premium())

## 2. Почему диаграмма внутренней стоимости врёт

Построим обычную выплату на экспирации (все ноги по внутренней стоимости) **против** настоящей кривой
переоценки по модели на дату экспирации ближней ноги (`t_elapsed = 21/365`). Реальна только вторая
кривая: когда ближняя нога истекает, дальний колл на 45 DTE ещё жив и всё ещё несёт временную
стоимость.

In [ ]:
spots = np.linspace(80, 120, 121)
naive = payoff.pnl_curve(cal, spots)                       # НЕВЕРНО для смешанных экспираций
real = payoff.pnl_curve(cal, spots, t_elapsed=21/365, vol=0.26)  # корректно
fig, ax = plt.subplots()
ax.plot(spots, naive, "--", label="наивная внутренняя стоимость (вводит в заблуждение)")
ax.plot(spots, real, label="pnl_at на экспирации ближней ноги (корректно)")
ax.axhline(0, color="k", lw=0.8); ax.axvline(100, color="grey", lw=0.6)
ax.set_xlabel("спот"); ax.set_ylabel("P&L $"); ax.legend(); ax.set_title("Календарь: палатка настоящая, V фальшивая")

Корректная кривая — это знакомая **палатка** с вершиной на страйке (100). Подтвердим вершину
численно в трёх точках спота.

In [ ]:
for s in (90, 100, 110):
    print(s, round(payoff.pnl_at(cal, s, t_elapsed=21/365, vol=0.26), 2))

## 3. Календарь длинный по веге

Используем `analyzer.scenario_grid` с `vol_shift`, чтобы подвинуть IV вверх и вниз, удерживая спот на
100 и шагнув на 10 дней вперёд. Календарь должен зарабатывать, когда волатильность **растёт** (в
сумме длинная вега).

In [ ]:
grid = analyzer.scenario_grid(
    cal, spots=[100.0], days_forward=[10], vol=0.26,
    vol_shift=[-0.03, 0.0, +0.03],
)
grid[["spot", "days_forward", "vol", "pnl"]]

Наибольший `pnl` — при наибольшей `vol`; именно это и значит «длинная вега». Здесь же и
предупреждение: широкий обвал волатильности, задевающий дальнюю ногу, календарю вредит.

## 4. Снимок греков позиции

`greeks.position_greeks` подтверждает профиль на входе: почти дельта-нейтрально, длинная тета
(положительная), длинная вега (положительная).

In [ ]:
g = greeks.position_greeks(cal, SPOT, vol=0.26)
print("дельта", round(g.delta, 2), "тета", round(g.theta, 2), "вега", round(g.vega, 2))

## 5. Бычья колл-диагональ

Покупаем 100-й колл на 45 DTE (3.91), продаём 105-й колл на 21 DTE (0.84). Разные страйки добавляют
направленный уклон; короткая нога субсидирует дебет.

In [ ]:
diag = strategies.diagonal_spread(
    "call", short=(105.0, 0.84), long=(100.0, 3.91),
    short_expiry=21/365, long_expiry=45/365,
)
print(diag.describe())
print("чистый дебет $:", diag.net_premium())

Сравним настоящую кривую диагонали на экспирации ближней ноги с календарём, чтобы увидеть
направленный перекос. Обе конструкции — со смешанными экспирациями, поэтому мы накладываем их
`pnl_curve` при `t_elapsed=21/365` (и никогда не берём `plot_compare`, который нарисовал бы
обманчивую выплату по внутренней стоимости).

In [ ]:
spots2 = np.linspace(88, 116, 141)
fig, ax = plt.subplots()
ax.plot(spots2, payoff.pnl_curve(cal, spots2, t_elapsed=21/365, vol=0.26), label="календарь (симметричный)")
ax.plot(spots2, payoff.pnl_curve(diag, spots2, t_elapsed=21/365, vol=0.26), label="колл-диагональ (скошена вверх)")
ax.axhline(0, color="k", lw=.7); ax.axvline(100, color="grey", lw=.6)
ax.legend(); ax.set_title("Календарь против колл-диагонали на экспирации ближней ноги")

## 6. «Бедняцкий покрытый колл»

Покупаем глубоко ITM долгосрочный колл (страйк 82.5, 180 DTE) в роли синтетической акции, продаём
против него ближний OTM-колл (105, 21 DTE). Длинную ногу оцениваем по BSM с IV из цепочки.

In [ ]:
iv_82 = chain[(chain.kind=="call") & (chain.strike==82.5) & (chain.expiry_days==180)].iv.iloc[0]
long_prem = pricing.bsm_price("call", SPOT, 82.5, 180/365, iv_82)
pmcc = strategies.poor_mans_covered_call(
    long_call=(82.5, round(long_prem, 2)), short_call=(105.0, 0.84),
    long_expiry=180/365, short_expiry=21/365,
)
print("модельная цена длинного колла:", round(long_prem, 2))
analyzer.summarize(pmcc, SPOT, vol=iv_82)

Прочитайте `max_profit`, `max_loss` и `net_premium` из сводки. PMCC работает только тогда,
когда повторяющиеся кредиты от коротких коллов стачивают дебетовую базу на протяжении многих циклов —
одного цикла почти никогда не хватает.

## Эксперименты

1. Перенесите страйк календаря на 105 (продать 105@21 = 0.84, купить 105@45 = 1.85). Как сместится
   палатка и какой направленный взгляд выражает OTM-календарь?
2. В вега-сетке замените `days_forward` на `[0, 10, 20]`. Посмотрите, как палатка растёт по мере
   приближения экспирации ближней ноги, — и отметьте гамма-риск последних дней.
3. Расширьте короткий страйк диагонали до 110@21 (0.18). Насколько меньше стало субсидии и сколько
   пространства для роста вы за это купили?
4. В PMCC возьмите длинный страйк 90 вместо 82.5. Вредит ли вашей тете более крупная временная
   стоимость менее глубокой длинной ноги? Сравните тету `position_greeks` для обоих вариантов.
5. Перезапустите календарь с `vol=0.20` везде. Более низкий режим волатильности сжимает палатку —
   почему?